# GKR Demo (PyTorch, Colab Ready)

This notebook is a standalone PyTorch implementation of GKR with an inducing-variable GP option for faster manifold fitting.

It does **not** depend on local project files, so you can run it directly in Google Colab.

In [ ]:
# Colab dependency setup
# If you already have a compatible runtime, this is safe to re-run.
%pip install -q --upgrade pip
%pip install -q numpy matplotlib scikit-learn gpytorch

In [ ]:
import importlib
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])

print("Torch is available.")

In [ ]:
from __future__ import annotations

import copy
import math
import warnings
from dataclasses import dataclass
from typing import Optional

import gpytorch
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.covariance import GraphicalLasso


def plot_cov_ellipse(mean, cov, ax, n_std=2.0, **kwargs):
    eigenvals, eigenvecs = np.linalg.eigh(cov)
    theta = np.arctan2(eigenvecs[1, 0], eigenvecs[0, 0])
    width, height = 2 * n_std * np.sqrt(np.clip(eigenvals, a_min=0.0, a_max=None))
    ellipse = Ellipse(
        xy=mean,
        width=width,
        height=height,
        angle=np.degrees(theta),
        **kwargs,
    )
    ax.add_patch(ellipse)


TensorLike = np.ndarray | torch.Tensor

In [ ]:
def _ensure_2d_tensor(
    value: TensorLike,
    *,
    dtype: torch.dtype,
    device: torch.device,
) -> torch.Tensor:
    tensor = torch.as_tensor(value, dtype=dtype, device=device)
    if tensor.ndim == 1:
        tensor = tensor.unsqueeze(-1)
    if tensor.ndim != 2:
        raise ValueError(f"Expected a 2D array, got shape {tuple(tensor.shape)}.")
    return tensor


def _circular_diff(diff: torch.Tensor, periods: Optional[float | list[Optional[float]]]) -> torch.Tensor:
    if periods is None:
        return diff

    if np.isscalar(periods):
        period = float(periods)
        return torch.sin(torch.pi * diff / period).pow(2)

    diff = diff.clone()
    for dim, period in enumerate(periods):
        if period is not None:
            diff[..., dim] = torch.sin(torch.pi * diff[..., dim] / float(period)).pow(2)
    return diff


def gaussian_log_likelihood(
    responses: torch.Tensor,
    covariance: torch.Tensor,
    *,
    diag_factor: float = 1e-5,
) -> torch.Tensor:
    responses = torch.as_tensor(responses, dtype=covariance.dtype, device=covariance.device)
    eye = torch.eye(covariance.shape[-1], dtype=covariance.dtype, device=covariance.device)
    stabilized = covariance + eye.unsqueeze(0) * diag_factor
    chol = torch.linalg.cholesky(stabilized)
    log_det = 2.0 * torch.log(torch.diagonal(chol, dim1=-2, dim2=-1)).sum(dim=-1)
    solved = torch.linalg.solve_triangular(chol, responses.unsqueeze(-1), upper=False)
    quadratic = solved.square().sum(dim=(-2, -1))
    return -0.5 * torch.mean(log_det + quadratic)


def _build_dimension_kernel(period: Optional[float], active_dim: int) -> gpytorch.kernels.Kernel:
    if period is None:
        return gpytorch.kernels.RBFKernel(active_dims=(active_dim,))

    kernel = gpytorch.kernels.PeriodicKernel(active_dims=(active_dim,))
    kernel.period_length = float(period)
    return kernel


def create_kernel(
    *,
    circular_period: Optional[float | list[Optional[float]]],
    input_dim: int,
) -> gpytorch.kernels.Kernel:
    if circular_period is None:
        base = gpytorch.kernels.RBFKernel(ard_num_dims=input_dim)
        return gpytorch.kernels.ScaleKernel(base)

    if np.isscalar(circular_period):
        kernels = [
            _build_dimension_kernel(float(circular_period), active_dim)
            for active_dim in range(input_dim)
        ]
    elif isinstance(circular_period, list):
        if len(circular_period) != input_dim:
            raise ValueError("Length of circular_period must match input_dim.")
        kernels = [_build_dimension_kernel(period, active_dim) for active_dim, period in enumerate(circular_period)]
    else:
        raise ValueError("Invalid input for circular_period.")

    combined = kernels[0]
    for kernel in kernels[1:]:
        combined = combined * kernel
    return gpytorch.kernels.ScaleKernel(combined)


class _ExactGPModel(gpytorch.models.ExactGP):
    def __init__(
        self,
        train_x: torch.Tensor,
        train_y: torch.Tensor,
        likelihood: gpytorch.likelihoods.GaussianLikelihood,
        kernel: gpytorch.kernels.Kernel,
    ) -> None:
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = kernel

    def forward(self, x: torch.Tensor) -> gpytorch.distributions.MultivariateNormal:
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


class _SparseGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points: torch.Tensor, kernel: gpytorch.kernels.Kernel) -> None:
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            inducing_points.size(0)
        )
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True,
        )
        super().__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = kernel

    def forward(self, x: torch.Tensor) -> gpytorch.distributions.MultivariateNormal:
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


@dataclass
class IndependentExactGPRegressor:
    n_outputs: int
    circular_period: Optional[float | list[Optional[float]]] = None
    standardize: bool = True
    n_inducing: Optional[int] = None
    seperate_kernel: bool = False
    training_iter: int = 150
    lr: float = 0.1
    kernel: Optional[gpytorch.kernels.Kernel] = None
    dtype: torch.dtype = torch.float64
    device: Optional[str | torch.device] = None

    def __post_init__(self) -> None:
        self.device = torch.device(self.device or "cpu")
        self.output_mean_: Optional[torch.Tensor] = None
        self.output_std_: Optional[torch.Tensor] = None
        self.models_: list[gpytorch.models.GP] = []
        self.likelihoods_: list[gpytorch.likelihoods.GaussianLikelihood] = []

    def _kernel_for_output(self, input_dim: int) -> gpytorch.kernels.Kernel:
        if self.kernel is None:
            return create_kernel(circular_period=self.circular_period, input_dim=input_dim)
        return copy.deepcopy(self.kernel)

    def _select_inducing_inputs(self, train_x: torch.Tensor, n_inducing: int) -> torch.Tensor:
        if n_inducing >= train_x.shape[0]:
            return train_x.clone()
        indices = torch.linspace(
            0,
            train_x.shape[0] - 1,
            steps=n_inducing,
            device=train_x.device,
        ).round().long()
        return train_x.index_select(0, indices)

    def fit(self, train_x: TensorLike, train_y: TensorLike) -> None:
        train_x = _ensure_2d_tensor(train_x, dtype=self.dtype, device=self.device)
        train_y = _ensure_2d_tensor(train_y, dtype=self.dtype, device=self.device)

        if train_y.shape[1] != self.n_outputs:
            raise ValueError("Number of output dimensions does not match n_outputs.")

        if self.n_inducing is not None:
            if not isinstance(self.n_inducing, int):
                raise ValueError("n_inducing must be an integer when provided.")
            if self.n_inducing < 1:
                raise ValueError("n_inducing must be a positive integer when provided.")

        if self.standardize:
            self.output_mean_ = train_y.mean(dim=0)
            self.output_std_ = train_y.std(dim=0).clamp_min(1e-8)
            normalized_y = (train_y - self.output_mean_) / self.output_std_
        else:
            self.output_mean_ = torch.zeros(train_y.shape[1], dtype=self.dtype, device=self.device)
            self.output_std_ = torch.ones(train_y.shape[1], dtype=self.dtype, device=self.device)
            normalized_y = train_y

        self.models_ = []
        self.likelihoods_ = []

        for output_idx in range(self.n_outputs):
            likelihood = gpytorch.likelihoods.GaussianLikelihood().to(self.device, self.dtype)
            kernel = self._kernel_for_output(train_x.shape[1]).to(self.device, self.dtype)
            if self.n_inducing is None:
                model: gpytorch.models.GP = _ExactGPModel(
                    train_x=train_x,
                    train_y=normalized_y[:, output_idx].contiguous(),
                    likelihood=likelihood,
                    kernel=kernel,
                ).to(self.device, self.dtype)
            else:
                n_inducing = min(int(self.n_inducing), train_x.shape[0])
                inducing_inputs = self._select_inducing_inputs(train_x, n_inducing)
                model = _SparseGPModel(
                    inducing_points=inducing_inputs,
                    kernel=kernel,
                ).to(self.device, self.dtype)

            model.train()
            likelihood.train()

            optimizer = torch.optim.Adam(model.parameters(), lr=self.lr)
            if self.n_inducing is None:
                mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
            else:
                mll = gpytorch.mlls.VariationalELBO(
                    likelihood,
                    model,
                    num_data=train_x.shape[0],
                )

            for _ in range(self.training_iter):
                optimizer.zero_grad()
                output = model(train_x)
                loss = -mll(output, normalized_y[:, output_idx])
                loss.backward()
                optimizer.step()

            self.models_.append(model)
            self.likelihoods_.append(likelihood)

    def predict(self, query: TensorLike) -> tuple[torch.Tensor, torch.Tensor]:
        query = _ensure_2d_tensor(query, dtype=self.dtype, device=self.device)
        means = []
        variances = []

        for model, likelihood in zip(self.models_, self.likelihoods_):
            model.eval()
            likelihood.eval()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=gpytorch.utils.warnings.GPInputWarning)
                with torch.no_grad(), gpytorch.settings.fast_pred_var():
                    posterior = model(query)
            means.append(posterior.mean.unsqueeze(-1))
            variances.append(posterior.variance.unsqueeze(-1))

        mean = torch.cat(means, dim=1)
        variance = torch.cat(variances, dim=1)
        mean = mean * self.output_std_ + self.output_mean_
        variance = variance * self.output_std_.pow(2)
        return mean, variance


class TorchKernelCovariance(torch.nn.Module):
    def __init__(
        self,
        n_input: int,
        n_output: int,
        circular_period: Optional[float | list[Optional[float]]] = None,
        *,
        diag_factor: float = 1e-6,
        dtype: torch.dtype = torch.float64,
        device: Optional[str | torch.device] = None,
    ) -> None:
        super().__init__()
        self.n_input = n_input
        self.n_output = n_output
        self.circular_period = circular_period
        self.diag_factor = diag_factor
        self.dtype = dtype
        self.device = torch.device(device or "cpu")

        initial = torch.randn(n_input, n_input, dtype=dtype, device=self.device)
        self.kernel_prec_L = torch.nn.Parameter(torch.tril(initial))

        self.register_buffer("train_responses", torch.empty(0, n_output, dtype=dtype, device=self.device))
        self.register_buffer("train_inputs", torch.empty(0, n_input, dtype=dtype, device=self.device))

    def fit(self, responses: TensorLike, inputs: TensorLike) -> None:
        responses = _ensure_2d_tensor(responses, dtype=self.dtype, device=self.device)
        inputs = _ensure_2d_tensor(inputs, dtype=self.dtype, device=self.device)
        self.train_responses = responses
        self.train_inputs = inputs

    def _precision_matrix(self) -> torch.Tensor:
        lower = torch.tril(self.kernel_prec_L)
        return lower @ lower.transpose(-1, -2)

    def predict_cov(self, query: TensorLike, pred_batch_size: int = 1000) -> torch.Tensor:
        if self.train_inputs.numel() == 0:
            raise RuntimeError("Kernel covariance estimator must be fit before predict_cov is called.")

        query = _ensure_2d_tensor(query, dtype=self.dtype, device=self.device)
        precision = self._precision_matrix()
        n_query = query.shape[0]

        cov_pred = torch.zeros(n_query, self.n_output, self.n_output, dtype=self.dtype, device=self.device)
        kernel_sum_total = torch.zeros(n_query, dtype=self.dtype, device=self.device)

        for start in range(0, self.train_inputs.shape[0], pred_batch_size):
            end = min(start + pred_batch_size, self.train_inputs.shape[0])
            batch_inputs = self.train_inputs[start:end]
            batch_responses = self.train_responses[start:end]

            diff = batch_inputs.unsqueeze(1) - query.unsqueeze(0)
            diff = _circular_diff(diff, self.circular_period)

            diff_prec = torch.einsum("bqi,ij,bqj->bq", diff, precision, diff)
            kernel_matrix = torch.exp(-diff_prec)
            kernel_sum_total = kernel_sum_total + kernel_matrix.sum(dim=0)

            gram = torch.einsum("bi,bj->bij", batch_responses, batch_responses)
            cov_pred = cov_pred + torch.einsum("bq,bij->qij", kernel_matrix, gram)

        cov_pred = cov_pred / kernel_sum_total.clamp_min(1e-12).view(-1, 1, 1)
        eye = torch.eye(self.n_output, dtype=self.dtype, device=self.device)
        return cov_pred + eye.unsqueeze(0) * self.diag_factor

    def forward(self, query: TensorLike, pred_batch_size: int = 1000) -> torch.Tensor:
        return self.predict_cov(query, pred_batch_size=pred_batch_size)


class GKRRegressor:
    def __init__(
        self,
        n_input: int,
        n_output: int,
        circular_period: Optional[float | list[Optional[float]]] = None,
        *,
        fit_valid_split: float = 0.3,
        learning_rate: float = 0.1,
        n_epochs: int = 100,
        gpr_params: Optional[dict] = None,
        cov_fit_batch_size: int = 3000,
        kernel_params: Optional[dict] = None,
        dtype: torch.dtype = torch.float64,
        device: Optional[str | torch.device] = None,
        random_state: Optional[int] = 0,
    ) -> None:
        self.n_input = n_input
        self.n_output = n_output
        self.circular_period = circular_period
        self.fit_valid_split = fit_valid_split
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.cov_fit_batch_size = cov_fit_batch_size
        self.dtype = dtype
        self.device = torch.device(device or "cpu")
        self.random_state = random_state

        gpr_params = dict(gpr_params or {})
        kernel_params = dict(kernel_params or {})

        self.mean_model = IndependentExactGPRegressor(
            n_outputs=n_output,
            circular_period=circular_period,
            dtype=dtype,
            device=self.device,
            **gpr_params,
        )
        self.covariance_model = TorchKernelCovariance(
            n_input=n_input,
            n_output=n_output,
            circular_period=circular_period,
            dtype=dtype,
            device=self.device,
            **kernel_params,
        )

        self.responses_: Optional[torch.Tensor] = None
        self.inputs_: Optional[torch.Tensor] = None
        self.loss_history_: list[float] = []

        self._generator = torch.Generator(device="cpu")
        if random_state is not None:
            self._generator.manual_seed(int(random_state))

    def _split_train_valid(self, responses: torch.Tensor, inputs: torch.Tensor) -> tuple[torch.Tensor, ...]:
        batch_size = responses.shape[0]
        if batch_size < 2:
            return responses, responses, inputs, inputs

        n_valid = max(1, int(round(batch_size * self.fit_valid_split)))
        n_valid = min(n_valid, batch_size - 1)
        indices = torch.randperm(batch_size, generator=self._generator)
        valid_idx = indices[:n_valid].to(inputs.device)
        train_idx = indices[n_valid:].to(inputs.device)
        return responses[train_idx], responses[valid_idx], inputs[train_idx], inputs[valid_idx]

    def fit(self, responses: TensorLike, inputs: TensorLike, fit_cov: bool = True) -> list[float]:
        responses_t = _ensure_2d_tensor(responses, dtype=self.dtype, device=self.device)
        inputs_t = _ensure_2d_tensor(inputs, dtype=self.dtype, device=self.device)

        if inputs_t.shape[1] != self.n_input:
            raise ValueError("Input feature dimension does not match n_input.")
        if responses_t.shape[1] != self.n_output:
            raise ValueError("Response dimension does not match n_output.")

        self.responses_ = responses_t
        self.inputs_ = inputs_t

        self.mean_model.fit(inputs_t, responses_t)
        mean_pred, _ = self.mean_model.predict(inputs_t)
        residuals = responses_t - mean_pred

        if not fit_cov:
            self.covariance_model.fit(residuals, inputs_t)
            self.loss_history_ = []
            return self.loss_history_

        optimizer = torch.optim.Adam(self.covariance_model.parameters(), lr=self.learning_rate)
        self.loss_history_ = []

        for _ in range(self.n_epochs):
            epoch_loss = 0.0
            permutation = torch.randperm(residuals.shape[0], generator=self._generator).to(self.device)

            for start in range(0, residuals.shape[0], self.cov_fit_batch_size):
                batch_idx = permutation[start : start + self.cov_fit_batch_size]
                if batch_idx.numel() == 0:
                    continue

                batch_responses = residuals[batch_idx]
                batch_inputs = inputs_t[batch_idx]
                r_train, r_valid, x_train, x_valid = self._split_train_valid(batch_responses, batch_inputs)

                optimizer.zero_grad()
                self.covariance_model.fit(r_train, x_train)
                cov_pred = self.covariance_model.predict_cov(x_valid)
                loss = -gaussian_log_likelihood(r_valid, cov_pred)
                loss.backward()
                optimizer.step()
                epoch_loss += float(loss.detach().cpu())

            self.loss_history_.append(epoch_loss)

        self.covariance_model.fit(residuals, inputs_t)
        return self.loss_history_

    def predict(
        self,
        query: TensorLike,
        *,
        return_cov: bool = True,
        with_GLASSO: Optional[float] = None,
    ) -> tuple[np.ndarray, Optional[np.ndarray]]:
        query_t = _ensure_2d_tensor(query, dtype=self.dtype, device=self.device)
        mean_pred, _ = self.mean_model.predict(query_t)
        mean_np = mean_pred.detach().cpu().numpy()

        if not return_cov:
            return mean_np, None

        cov_pred = self.covariance_model.predict_cov(query_t).detach().cpu().numpy()
        if with_GLASSO is not None:
            cov_glasso = np.zeros_like(cov_pred)
            for idx, cov_matrix in enumerate(cov_pred):
                glasso = GraphicalLasso(alpha=with_GLASSO, assume_centered=True)
                glasso.fit(cov_matrix)
                cov_glasso[idx] = glasso.covariance_
            cov_pred = cov_glasso

        return mean_np, cov_pred


def generate_circular_dataset(
    n_data: int,
    *,
    radius: float = 1.0,
    noise_factor: float = 0.1,
    dtype: torch.dtype = torch.float64,
    device: Optional[str | torch.device] = None,
    seed: Optional[int] = 0,
    as_tensor: bool = False,
) -> tuple[TensorLike, TensorLike]:
    device = torch.device(device or "cpu")
    generator = torch.Generator(device="cpu")
    if seed is not None:
        generator.manual_seed(int(seed))

    angles = torch.linspace(0.0, 2.0 * math.pi, n_data, dtype=dtype, device=device).unsqueeze(-1)
    response = torch.cat((radius * torch.cos(angles), radius * torch.sin(angles)), dim=1)
    noise = torch.randn(response.shape, generator=generator, dtype=dtype).to(device) * noise_factor
    response_noisy = response + noise

    if as_tensor:
        return response_noisy, angles
    return response_noisy.cpu().numpy(), angles.cpu().numpy()


print("Standalone GKR implementation loaded.")

## Generate toy circular data

In [ ]:
n_data = 100
response_noisy, labels = generate_circular_dataset(
    n_data,
    radius=1.0,
    noise_factor=0.2,
    seed=0,
)

plt.figure(figsize=(6, 6))
plt.scatter(response_noisy[:, 0], response_noisy[:, 1], c=labels.ravel(), cmap="viridis")
plt.title("Noisy Circular Data")
plt.xlabel("X")
plt.ylabel("Y")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

## Fit GKR with inducing-variable GP

Set `n_inducing` in `gpr_params` to enable sparse variational GP for the mean manifold model.

In [ ]:
gkr = GKRRegressor(
    n_input=labels.shape[1],
    n_output=response_noisy.shape[1],
    circular_period=2 * np.pi,
    n_epochs=5,
    cov_fit_batch_size=64,
    random_state=0,
    gpr_params={
        "training_iter": 75,
        "lr": 0.12,
        "n_inducing": 20,
    },
)

loss_history = gkr.fit(response_noisy, labels, fit_cov=True)
print(f"Training finished. Epoch losses: {loss_history}")

## Predict manifold and local covariance

In [ ]:
label_pred = np.linspace(0, 2 * np.pi, 100).reshape(-1, 1)
response_pred, response_cov = gkr.predict(label_pred)

print("Predicted mean shape:", response_pred.shape)
print("Predicted covariance shape:", response_cov.shape)
print("All finite:", np.isfinite(response_pred).all() and np.isfinite(response_cov).all())

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(response_noisy[:, 0], response_noisy[:, 1], c=labels.ravel(), cmap="viridis", label="Noisy Samples")
plt.plot(response_pred[:, 0], response_pred[:, 1], "r-", linewidth=2, label="Predicted Manifold")

for idx in range(0, len(response_pred), 8):
    plot_cov_ellipse(
        response_pred[idx],
        response_cov[idx],
        plt.gca(),
        alpha=0.2,
        color="red",
    )

plt.title("GKR Manifold And Local Covariance")
plt.xlabel("X")
plt.ylabel("Y")
plt.gca().set_aspect("equal", adjustable="box")
plt.legend()
plt.show()

## Optional: save outputs in Colab filesystem

In [ ]:
np.savez(
    "gkr_demo_outputs.npz",
    response_noisy=response_noisy,
    labels=labels,
    response_pred=response_pred,
    response_cov=response_cov,
)
print("Saved gkr_demo_outputs.npz")